# Foundations of Prompt Engineering for SQL Workflows

**Scenario:** A regional healthcare network has seen more missed appointments and inconsistent follow-up care. This lab uses the **Identify → Assemble → Execute → Verify** workflow to design prompts that guide an AI assistant to generate SQL against `healthcare_operations.db`.

**AI assistant used:** Claude (Anthropic). Each prompt below was submitted as written, and the SQL it returned is saved without manual edits.

**Scope chosen:** *Moderate.* The question is about patients with multiple missed **or** canceled appointments.

## Setup and Validation Checks

These checks confirm the tables, row counts, date boundaries and exact status values **before** any AI-generated SQL is trusted.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("healthcare_operations.db")

# Confirm tables and expected row counts
expected = {"patients": 50, "appointments": 120, "providers": 10, "clinics": 3, "reporting_context": 1}
row_counts = pd.DataFrame(
    [{"table_name": t,
      "row_count": pd.read_sql(f"SELECT COUNT(*) AS n FROM {t};", conn).iloc[0, 0],
      "expected": n} for t, n in expected.items()]
)
row_counts

,table_name,row_count,expected
0,patients,50,50
1,appointments,120,120
2,providers,10,10
3,clinics,3,3
4,reporting_context,1,1


In [2]:
# Fixed reporting dates (use these instead of DATE('now'))
pd.read_sql("SELECT * FROM reporting_context;", conn).T

,0
context_id,1
reporting_date,2025-06-29
data_start_date,2025-01-06
data_end_date,2025-06-29
recent_30_day_start,2025-05-30
recent_60_day_start,2025-04-30
recent_3_month_start,2025-03-29
recent_6_month_start,2024-12-29
primary_date_column,appointments.appointment_date
note,Use reporting_date instead of date('now') beca...


In [3]:
# Exact status spelling and the actual date range of the appointment data
display(pd.read_sql("""
SELECT appointment_status, COUNT(*) AS row_count
FROM appointments
GROUP BY appointment_status
ORDER BY row_count DESC;
""", conn))
pd.read_sql("SELECT MIN(appointment_date) AS min_date, MAX(appointment_date) AS max_date FROM appointments;", conn)

,appointment_status,row_count
0,completed,87
1,no_show,23
2,canceled,10


,min_date,max_date
0,2025-01-06,2025-06-29


## Task 1: Identify (Clarify the Business Problem)

**Business question:** Which patients have missed two or more appointments, either as no-shows or cancellations, during the most recent 3-month reporting window, and how large a share of their scheduled visits did they miss?

**Why it matters:** Missed appointments leave provider time unused and lower clinic efficiency. They also interrupt follow-up care, which can worsen patient outcomes. A list of repeat-miss patients gives the care team a short list for reminder calls, rescheduling outreach or scheduling changes such as confirmation calls or double-booking rules.

**Key definitions:**
- **Missed appointment:** `appointment_status IN ('no_show', 'canceled')`. Both statuses represent a scheduled visit that did not happen. They are also counted separately, because a no-show (no notice) is a bigger operational problem than a cancellation.
- **Recent window:** `appointment_date` between `reporting_context.recent_3_month_start` (2025-03-29) and `reporting_context.reporting_date` (2025-06-29). The dataset is fictional and historical, so `DATE('now')` must not be used.
- **High-risk patient:** a patient with **2 or more** missed appointments in the recent window.

**Expected output:** one row per high-risk patient with columns `patient_id`, `patient_name`, `insurance_type`, `total_appointments`, `no_show_count`, `canceled_count`, `missed_count`, `missed_rate_pct` and `last_appointment_date`, sorted with the most missed appointments first.

## Task 2: Assemble (Break Down the Query Requirements)

| Requirement | Plan |
|---|---|
| **Tables** | `patients`, `appointments`, `reporting_context` (date boundaries only, so no join) |
| **Columns** | `patients`: `patient_id`, `patient_name`, `insurance_type` · `appointments`: `appointment_id`, `patient_id`, `appointment_date`, `appointment_status` · `reporting_context`: `recent_3_month_start`, `reporting_date` |
| **Joins** | `patients.patient_id = appointments.patient_id` (inner join, since only patients with appointments matter) |
| **Filters** | `appointment_date` between `recent_3_month_start` and `reporting_date`, read from `reporting_context` with scalar subqueries |
| **Status logic** | Count `'no_show'` and `'canceled'` separately and together. Keep `completed` rows so the total and the missed rate are correct; filtering them out in `WHERE` would make every rate 100% |
| **Grouping / counting** | `GROUP BY` patient, `COUNT` of appointments, conditional `SUM(CASE ...)` for each status |
| **Thresholds** | `HAVING` missed count `>= 2` |
| **Sorting** | `missed_count DESC`, then `missed_rate_pct DESC`, then `patient_name` as a tiebreaker |
| **Final output** | The 9 columns listed in Task 1 |

The validation checks above confirm that every table and column exists, that missed visits are stored as `no_show` (with an underscore), and that the appointment data runs from 2025-01-06 to 2025-06-29.

## Task 3: Execute (Build the Initial Prompt)

```text
Task:
Write a SQL query that identifies high-risk patients (patients with repeated missed
appointments in the last 3 months) so the clinic operations team can prioritize reminder
calls and rescheduling outreach.

Schema:
patients(patient_id, patient_name, insurance_type)
appointments(appointment_id, patient_id, appointment_date, appointment_status)
Relationship: patients.patient_id = appointments.patient_id

Constraints:
- Only include appointments from the last 3 months.
- A missed appointment is an appointment with a status of no-show or canceled.
- A high-risk patient has 2 or more missed appointments in that period.
- total_appointments should count all of the patient's appointments in the period,
  not only the missed ones.

Output:
One row per high-risk patient with columns: patient_id, patient_name, insurance_type,
total_appointments, missed_count, missed_rate_pct (percent of appointments missed,
rounded to 1 decimal). Sort by missed_count descending.
```

*Looking back:* this prompt covers the business intent, schema, join, definitions and output columns. But it describes the time window and the statuses in everyday language ("last 3 months", "no-show") instead of the exact database values. The review in Task 5 shows how that gap played out.

## Task 4: Execute (Generate SQL with an AI Assistant)

The initial prompt above was submitted to the AI assistant. The SQL below is saved **exactly as generated**, with no manual edits.

```sql
SELECT
    p.patient_id,
    p.patient_name,
    p.insurance_type,
    COUNT(a.appointment_id) AS total_appointments,
    SUM(CASE WHEN a.appointment_status IN ('no-show', 'canceled') THEN 1 ELSE 0 END) AS missed_count,
    ROUND(SUM(CASE WHEN a.appointment_status IN ('no-show', 'canceled') THEN 1 ELSE 0 END) * 100.0 / COUNT(a.appointment_id), 1) AS missed_rate_pct
FROM patients p
JOIN appointments a
    ON p.patient_id = a.patient_id
WHERE a.appointment_date >= DATE('now', '-3 months')
GROUP BY p.patient_id, p.patient_name, p.insurance_type
HAVING missed_count >= 2
ORDER BY missed_count DESC;
```

In [4]:
initial_sql = """
SELECT
    p.patient_id,
    p.patient_name,
    p.insurance_type,
    COUNT(a.appointment_id) AS total_appointments,
    SUM(CASE WHEN a.appointment_status IN ('no-show', 'canceled') THEN 1 ELSE 0 END) AS missed_count,
    ROUND(SUM(CASE WHEN a.appointment_status IN ('no-show', 'canceled') THEN 1 ELSE 0 END) * 100.0 / COUNT(a.appointment_id), 1) AS missed_rate_pct
FROM patients p
JOIN appointments a
    ON p.patient_id = a.patient_id
WHERE a.appointment_date >= DATE('now', '-3 months')
GROUP BY p.patient_id, p.patient_name, p.insurance_type
HAVING missed_count >= 2
ORDER BY missed_count DESC;
"""
initial_result = pd.read_sql(initial_sql, conn)
print(f"Initial SQL returned {len(initial_result)} rows")
initial_result

Initial SQL returned 0 rows


,patient_id,patient_name,insurance_type,total_appointments,missed_count,missed_rate_pct


## Task 5: Verify (Review the SQL Output)

The query ran without errors but returned **0 rows**. The filter-by-filter check below shows why.

In [5]:
pd.read_sql("""
SELECT 'all appointments' AS step, COUNT(*) AS row_count FROM appointments
UNION ALL
SELECT 'AI date filter: >= DATE(''now'', ''-3 months'')', COUNT(*)
FROM appointments WHERE appointment_date >= DATE('now', '-3 months')
UNION ALL
SELECT 'anchored date filter: >= recent_3_month_start', COUNT(*)
FROM appointments WHERE appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
UNION ALL
SELECT 'AI status values: IN (''no-show'', ''canceled'')', COUNT(*)
FROM appointments WHERE appointment_status IN ('no-show', 'canceled')
UNION ALL
SELECT 'actual status values: IN (''no_show'', ''canceled'')', COUNT(*)
FROM appointments WHERE appointment_status IN ('no_show', 'canceled');
""", conn)

,step,row_count
0,all appointments,120
1,"AI date filter: >= DATE('now', '-3 months')",0
2,anchored date filter: >= recent_3_month_start,58
3,"AI status values: IN ('no-show', 'canceled')",10
4,"actual status values: IN ('no_show', 'canceled')",33


**Review notes**

| Check | Result | Notes |
|---|---|---|
| Correct tables | ✅ | Uses `patients` and `appointments` |
| Correct join | ✅ | `p.patient_id = a.patient_id` |
| Valid columns | ✅ | No invented columns, because the prompt listed the schema |
| Correct filters | ❌ | **Date:** `DATE('now', '-3 months')` points to 2026, but the data ends 2025-06-29, so every row is filtered out. **Status:** `'no-show'` (hyphen) does not exist; the stored value is `'no_show'`, so no-shows would never be counted even with a correct date window |
| Grouping / aggregation | ✅ / ⚠️ | Correct: `GROUP BY` patient, `COUNT` keeps completed visits in the total (as my constraint asked), and `SUM(CASE ...)` counts misses. Gap: no-shows and cancellations are merged into one number, which hides the more serious no-shows |
| Expected output | ⚠️ | The column names and order match my Output section, but the query returns 0 rows. It has no recency column to help with outreach, and the window has no upper bound |
| Matches business question | ❌ | Cannot answer it while it returns nothing |
| Made-up assumptions | ❌ | No invented tables or columns, but the AI *assumed* "last 3 months" meant relative to today and *guessed* the spelling of the status value. Both came from everyday wording in my prompt |
| Sorting | ⚠️ | No tiebreaker, so patients with equal counts come back in an arbitrary order |

**What worked:** listing the schema, the relationship and the output columns kept the AI from inventing tables or columns, and the overall shape (join, group, conditional count, `HAVING`, rounded rate) was correct.

**What needs improvement:** the prompt did not tell the AI which SQL dialect to use, that the dataset is historical, how to anchor the date window, or the exact stored status values. It also left out a tie-breaking sort and a way to tell no-shows apart from cancellations.

## Task 6: Verify (Refine the Prompt)

```text
Task:
You are writing a SQLite3 query for a healthcare network's clinic operations team. Identify
"high-risk" patients: patients with 2 or more missed appointments in the most recent 3-month
reporting window, so staff can prioritize reminder and rescheduling outreach.

Schema (use ONLY these tables and columns; do not invent others):
patients(patient_id, patient_name, insurance_type)
appointments(appointment_id, patient_id, appointment_date TEXT 'YYYY-MM-DD',
             appointment_status)   -- exact values: 'completed', 'canceled', 'no_show'
reporting_context(recent_3_month_start, reporting_date)  -- exactly 1 row; does NOT join
                                                          -- to other tables
Join: patients.patient_id = appointments.patient_id (INNER JOIN).

Constraints:
- SQLite dialect. Do NOT use DATE('now') or CURRENT_DATE. The data is historical. Define the
  window as appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
  AND appointment_date <= (SELECT reporting_date FROM reporting_context).
- Missed appointment = appointment_status IN ('no_show', 'canceled'), using these exact
  lowercase values with an underscore.
- total_appointments must count ALL appointments in the window (including 'completed'),
  so do not filter statuses in WHERE; use conditional aggregation (SUM(CASE ...)).
- Keep only patients with missed_count >= 2 (use HAVING).
- missed_rate_pct = 100.0 * missed_count / total_appointments, rounded to 1 decimal.

Output (exact column names, in this order):
patient_id, patient_name, insurance_type, total_appointments, no_show_count, canceled_count,
missed_count, missed_rate_pct, last_appointment_date (latest appointment_date in the window).
Sort by missed_count DESC, then missed_rate_pct DESC, then patient_name ASC.
Return a single SQL statement only.
```

**Revised AI-generated SQL** (saved exactly as generated):

```sql
WITH window_appointments AS (
    SELECT
        a.appointment_id,
        a.patient_id,
        a.appointment_date,
        a.appointment_status
    FROM appointments AS a
    WHERE a.appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
      AND a.appointment_date <= (SELECT reporting_date FROM reporting_context)
)
SELECT
    p.patient_id,
    p.patient_name,
    p.insurance_type,
    COUNT(w.appointment_id) AS total_appointments,
    SUM(CASE WHEN w.appointment_status = 'no_show' THEN 1 ELSE 0 END) AS no_show_count,
    SUM(CASE WHEN w.appointment_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_count,
    SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) AS missed_count,
    ROUND(
        100.0 * SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END)
        / COUNT(w.appointment_id),
        1
    ) AS missed_rate_pct,
    MAX(w.appointment_date) AS last_appointment_date
FROM patients AS p
JOIN window_appointments AS w
    ON p.patient_id = w.patient_id
GROUP BY p.patient_id, p.patient_name, p.insurance_type
HAVING SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) >= 2
ORDER BY missed_count DESC, missed_rate_pct DESC, p.patient_name ASC;
```

**What changed and where each improvement came from**

| Improvement | Category | Source |
|---|---|---|
| Stated SQLite dialect and banned `DATE('now')`; anchored the window to `reporting_context` with both a start and an end bound | Dialect clarity, Filter logic | **My review.** Found with the filter-by-filter check in Task 5 |
| Gave the exact stored status values (`'no_show'`, `'canceled'`) | Business definition, Filter logic | **My review.** Confirmed with the status value check in the validation cells |
| Listed the allowed status values in the schema, added date format, and told the AI to use *only* the listed columns | Schema clarity | **My review** |
| Spelled out conditional aggregation (`SUM(CASE ...)`, no status filter in `WHERE`) instead of only describing the total | Aggregation logic | **My review.** Added so a rewrite cannot move the status filter into `WHERE`, which would make every rate 100% |
| Split no-shows and cancellations into separate counts, and added `last_appointment_date` | Output formatting | **My review** (requirement) + **AI refinement** (the AI chose the `MAX(appointment_date)` implementation) |
| Exact column names, order and three-level sort | Output formatting | **My review** |
| CTE that pre-filters the window, and `HAVING` written with the full expression instead of an alias | SQL structure | **AI refinement.** Not requested; the AI chose this structure from the revised prompt |

## Task 7: Verify (Test the Revised SQL)

In [6]:
revised_sql = """
WITH window_appointments AS (
    SELECT
        a.appointment_id,
        a.patient_id,
        a.appointment_date,
        a.appointment_status
    FROM appointments AS a
    WHERE a.appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
      AND a.appointment_date <= (SELECT reporting_date FROM reporting_context)
)
SELECT
    p.patient_id,
    p.patient_name,
    p.insurance_type,
    COUNT(w.appointment_id) AS total_appointments,
    SUM(CASE WHEN w.appointment_status = 'no_show' THEN 1 ELSE 0 END) AS no_show_count,
    SUM(CASE WHEN w.appointment_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_count,
    SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) AS missed_count,
    ROUND(
        100.0 * SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END)
        / COUNT(w.appointment_id),
        1
    ) AS missed_rate_pct,
    MAX(w.appointment_date) AS last_appointment_date
FROM patients AS p
JOIN window_appointments AS w
    ON p.patient_id = w.patient_id
GROUP BY p.patient_id, p.patient_name, p.insurance_type
HAVING SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) >= 2
ORDER BY missed_count DESC, missed_rate_pct DESC, p.patient_name ASC;
"""
revised_result = pd.read_sql(revised_sql, conn)
print(f"Revised SQL returned {len(revised_result)} rows")
revised_result

Revised SQL returned 4 rows


,patient_id,patient_name,insurance_type,total_appointments,no_show_count,canceled_count,missed_count,missed_rate_pct,last_appointment_date
0,28,Brandon Carter,HMO,4,2,1,3,75.0,2025-06-05
1,17,Quinn Lewis,Medicaid,2,2,0,2,100.0,2025-04-25
2,39,Maya Stewart,Medicaid,3,2,0,2,66.7,2025-05-28
3,46,Thomas Bell,Medicare,3,2,0,2,66.7,2025-06-07


**Independent checks.** The same answer is rebuilt with pandas (no SQL aggregation), the patients are compared with the validation notebook's repeat no-show check, and the output columns are confirmed.

In [7]:
ctx = pd.read_sql("SELECT recent_3_month_start, reporting_date FROM reporting_context;", conn).iloc[0]
appts = pd.read_sql("SELECT * FROM appointments;", conn)

# 1) Rebuild the result in pandas
window = appts[(appts["appointment_date"] >= ctx["recent_3_month_start"]) &
               (appts["appointment_date"] <= ctx["reporting_date"])]
check = (window.assign(missed=window["appointment_status"].isin(["no_show", "canceled"]))
               .groupby("patient_id")["missed"].agg(["count", "sum"]))
pandas_high_risk = set(check[check["sum"] >= 2].index)
assert pandas_high_risk == set(revised_result["patient_id"]), "Mismatch with pandas rebuild"

# 2) The join must not drop or duplicate appointment rows
join_rows = pd.read_sql("""
SELECT COUNT(*) AS n FROM appointments a JOIN patients p ON a.patient_id = p.patient_id
WHERE a.appointment_date BETWEEN (SELECT recent_3_month_start FROM reporting_context)
                             AND (SELECT reporting_date FROM reporting_context);
""", conn).iloc[0, 0]
assert join_rows == len(window)

# 3) Exact output columns and internal consistency
assert list(revised_result.columns) == [
    "patient_id", "patient_name", "insurance_type", "total_appointments", "no_show_count",
    "canceled_count", "missed_count", "missed_rate_pct", "last_appointment_date"]
assert (revised_result["missed_count"] == revised_result["no_show_count"] + revised_result["canceled_count"]).all()
assert (revised_result["missed_count"] <= revised_result["total_appointments"]).all()

print(f"Appointments in window: {len(window)} | join preserved rows: {join_rows == len(window)}")
print("All checks passed: revised SQL matches the independent pandas rebuild.")

Appointments in window: 58 | join preserved rows: True
All checks passed: revised SQL matches the independent pandas rebuild.


In [8]:
# 4) Compare with the validation notebook's expected-output check (repeat no_show only)
pd.read_sql("""
SELECT p.patient_id, p.patient_name, COUNT(a.appointment_id) AS no_show_count
FROM patients AS p
JOIN appointments AS a ON p.patient_id = a.patient_id
WHERE a.appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
  AND a.appointment_status = 'no_show'
GROUP BY p.patient_id, p.patient_name
HAVING COUNT(a.appointment_id) >= 2
ORDER BY no_show_count DESC, p.patient_name;
""", conn)

,patient_id,patient_name,no_show_count
0,28,Brandon Carter,2
1,39,Maya Stewart,2
2,17,Quinn Lewis,2
3,46,Thomas Bell,2


**Test results**

| Question | Result |
|---|---|
| Does the SQL run successfully? | ✅ Yes. It runs in SQLite3 with no errors |
| Does the output match the business question? | ✅ Yes. It returns **4 high-risk patients**: Brandon Carter (3 missed, 75%), Quinn Lewis (2, 100%), Maya Stewart (2, 66.7%) and Thomas Bell (2, 66.7%). The pandas rebuild returns the same patients, and the join keeps all 58 appointments in the window |
| Any errors or issues remaining? | No errors. One limitation: in this window every high-risk patient has at least 2 no-shows, so the same 4 patients appear in the validation notebook's no-show-only check. Over the full 6-month dataset there are 7 patients with 2+ misses, so the window choice matters. Also, some patients have very few appointments (Quinn Lewis has 2), so the rate should be read alongside the count |
| Did the revised prompt improve the result? | ✅ Yes. It went from **0 rows** (wrong date anchor, wrong status spelling) to a correct, validated list. The separate no-show/cancel counts and `last_appointment_date` make the list usable for outreach |

## Task 8: Verify (Explain What I Learned)

**What was the most important improvement I made to my prompt?**
My most important improvement was grounding the prompt in the actual data: telling the AI to use `reporting_context` for the date window instead of "the last 3 months", and giving the exact stored status values (`'no_show'`, `'canceled'`). My first prompt helps to describe the problem in everyday language, and the AI filled the gaps with reasonable but wrong assumptions.

**How did that improvement affect the SQL output?**
The first query was valid SQL and ran without errors, but it returned 0 rows because `DATE('now')` pointed past the end of the data and `'no-show'` matched nothing. After the revision, the query returned 4 patients, and I confirmed them independently in pandas. Asking for separate no-show and cancel counts, a recency column and a defined sort order also made the result something the clinic team could act on.

**What did this lab teach me about prompt engineering for SQL workflows?**
A query that runs is not the same as a query that is correct. The AI can only be as precise as the prompt: vague terms like "recent" and "missed" become guesses. The Task / Schema / Constraints / Output structure made me define every business term and value up front, and the validation notebook (row counts, status values, filter-by-filter checks) was how I found the exact problems to fix.

**What would I do differently next time?**
One thing i'd do diffently would be running the validation checks *before* writing the first prompt and paste the exact category values and date boundaries into it from the start. I would also write down the expected row count or a few known answers ahead of time, so I can tell right away whether the AI's output is plausible.

In [9]:
conn.close()
print("Database connection closed.")

Database connection closed.
